<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/Course_Work/Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenization and Vectorization - Embeddings


## Bag of words

In [ ]:
import numpy as np

In [ ]:
# Sample documents
documents = [
    "data representation in computers",
    "data representation matters",
    "computers process data"
]


In [ ]:
# Step 1: Build vocabulary (unique words)
vocabulary = sorted(set(word for doc in documents for word in doc.split()))

In [ ]:
# Step 2: Create Bag of Words vectors
bow_vectors = []
for doc in documents:
    words = doc.split()
    bow_vectors.append([words.count(word) for word in vocabulary])

bow_vectors = np.array(bow_vectors)

In [ ]:
# Output
print("Vocabulary:", vocabulary)
print("\nBag of Words Matrix:")
print(bow_vectors)
print("\nMatrix Shape:", bow_vectors.shape)

Vocabulary: ['computers', 'data', 'in', 'matters', 'process', 'representation']

Bag of Words Matrix:
[[1 1 1 0 0 1]
 [0 1 0 1 0 1]
 [1 1 0 0 1 0]]

Matrix Shape: (3, 6)


## Using Scikit Learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
documents = ["This is the first document.", "This document is the second document.",
              "And this is the third one.", "Is this the first document?"]

In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

In [ ]:
print("Bag-of-Words Matrix:")
print(X.toarray())
print("Vocabulary (Feature Names):", feature_names)

Bag-of-Words Matrix:
[[0 1 1 1 0 0 1 0 1]
 [0 2 0 1 0 1 1 0 1]
 [1 0 0 1 1 0 1 1 1]
 [0 1 1 1 0 0 1 0 1]]
Vocabulary (Feature Names): ['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']


##  Word to Vector
static embeddings don't differentiate by context so river bank and blood bank - the bank has the same embeddings...

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.5 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

documents = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
    "This is a blood bank",
    "This is a river bank"
]

# Tokenize (lowercase, remove punctuation)
tokenized_docs = [simple_preprocess(doc) for doc in documents]

print(tokenized_docs)


[['this', 'is', 'the', 'first', 'document'], ['this', 'document', 'is', 'the', 'second', 'document'], ['and', 'this', 'is', 'the', 'third', 'one'], ['is', 'this', 'the', 'first', 'document'], ['this', 'is', 'blood', 'bank'], ['this', 'is', 'river', 'bank']]


In [ ]:
# Step-2 Train a word2Vec model
model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,   # embedding dimension
    window=5,          # context window size
    min_count=1,       # include all words (small dataset)
    workers=4,
    sg=1               # 1 = Skip‑gram, 0 = CBOW
)

In [ ]:
#Step -3 Access word embeddings
word = "bank"
embedding = model.wv[word]

print ("------")
print("Word embedding for",{word})
print(embedding)
print ("-------")
print("Embedding dimension:", embedding.shape)

------
Word embedding for {'bank'}
[-0.0071404   0.00124127 -0.00717811 -0.00224505  0.00372003  0.00583426
  0.00119842  0.00210314 -0.00411119  0.00722674 -0.00630827  0.00464812
 -0.00822157  0.00203686 -0.00497802 -0.00424851 -0.00310959  0.00565631
  0.00579953 -0.00497562  0.00077348 -0.00849743  0.00781132  0.00925909
 -0.00274286  0.00080038  0.0007468   0.00547895 -0.00860775  0.00058457
  0.00687076  0.00223203  0.0011249  -0.00932397  0.00848402 -0.00626535
 -0.00299296  0.00349447 -0.00077278  0.00141157  0.00178234 -0.00683023
 -0.0097267   0.00904234  0.00619926 -0.00691427  0.00340414  0.0002061
  0.00475467 -0.00712133  0.00402774  0.00434828  0.00995931 -0.00447461
 -0.00138953 -0.00731874 -0.00969972 -0.00908202 -0.00102295 -0.00650455
  0.00485067 -0.00616523  0.00251968  0.00073958 -0.00339281 -0.00097941
  0.00998107  0.00914766 -0.0044627   0.00908479 -0.00564286  0.00593207
 -0.00309782  0.00343242  0.00301781  0.0069018  -0.00237435  0.00877674
  0.0075909  -0.0

In [ ]:
#Step -4 Check the vocabulary
print("Vocabulary:")
print(list(model.wv.index_to_key))

Vocabulary:
['is', 'this', 'document', 'the', 'bank', 'first', 'river', 'blood', 'one', 'third', 'and', 'second']


In [ ]:
#Step-5 Similarity between words
similarity = model.wv.similarity("first", "second")
print("Similarity between 'first' and 'second':", similarity)

Similarity between 'first' and 'second': 0.16694683


## Contextual word embeddings - Transformers

In [ ]:
"""
Transformer component demonstration using a real sentence.

Pipeline:
Sentence
 → Tokenization
 → Token IDs
 → Token Embedding
 → Positional Encoding
 → Self-Attention
 → Multi-Head Attention
 → Feed-Forward Network
 → Stacked Transformer Layers
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# -------------------------
# 1. Input sentence
# -------------------------
sentence = "Transformers learn contextual word representations"

# Simple whitespace tokenization (educational purpose)
tokens = sentence.lower().split()

print("Tokens:", tokens)

# -------------------------
# 2. Vocabulary + Token IDs
# -------------------------
vocab = {word: idx for idx, word in enumerate(set(tokens))}
vocab_size = len(vocab)

token_ids = torch.tensor([[vocab[word] for word in tokens]])

print("Token IDs:", token_ids)

# -------------------------
# Configuration
# -------------------------
BATCH_SIZE = 1
SEQ_LEN = token_ids.shape[1]
D_MODEL = 64
NUM_HEADS = 8
NUM_LAYERS = 2

Tokens: ['transformers', 'learn', 'contextual', 'word', 'representations']
Token IDs: tensor([[4, 2, 0, 3, 1]])


In [ ]:
# -------------------------
# 3. Token Embedding
# Purpose: Convert tokens to vectors
# -------------------------
embedding_layer = nn.Embedding(vocab_size, D_MODEL)
token_embeddings = embedding_layer(token_ids)

print("Token Embeddings Shape:", token_embeddings.shape)
# (B, S, D)

Token Embeddings Shape: torch.Size([1, 5, 64])


In [ ]:
# -------------------------
# 4. Positional Encoding
# Purpose: Encode sequence order
# -------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

positional_encoding = PositionalEncoding(D_MODEL)
x = positional_encoding(token_embeddings)

In [ ]:
# -------------------------
# 5. Self-Attention
# Purpose: Context modeling
# -------------------------
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

    def forward(self, x):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(x.size(-1))
        weights = F.softmax(scores, dim=-1)

        return torch.matmul(weights, V)

self_attention = SelfAttention(D_MODEL)
x = self_attention(x)

print("Self-Attention Output Shape:", x.shape)

Self-Attention Output Shape: torch.Size([1, 5, 64])


In [ ]:
# -------------------------
# 6. Multi-Head Attention
# Purpose: Multiple meaning perspectives
# -------------------------
multi_head_attention = nn.MultiheadAttention(
    embed_dim=D_MODEL,
    num_heads=NUM_HEADS,
    batch_first=True
)

x, attn_weights = multi_head_attention(x, x, x)

print("Multi-Head Attention Output Shape:", x.shape)

Multi-Head Attention Output Shape: torch.Size([1, 5, 64])


In [ ]:
# -------------------------
# 7. Feed-Forward Network
# Purpose: Refinement
# -------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

ffn = FeedForward(D_MODEL)
x = ffn(x)

print("Feed-Forward Output Shape:", x.shape)

Feed-Forward Output Shape: torch.Size([1, 5, 64])


In [ ]:
# -------------------------
# 8. Transformer Layer
# -------------------------
class TransformerLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.mha = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True
        )
        self.ffn = FeedForward(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.mha(x, x, x)
        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)

        return x

In [ ]:
# -------------------------
# 9. Stacking Transformer Layers
# Purpose: Deep understanding
# -------------------------
layers = nn.ModuleList(
    [TransformerLayer(D_MODEL, NUM_HEADS) for _ in range(NUM_LAYERS)]
)

for layer in layers:
    x = layer(x)

print("Final Transformer Output Shape:", x.shape)

Final Transformer Output Shape: torch.Size([1, 5, 64])
